[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.3_enterprise_rag/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.3_enterprise_rag/lab.ipynb)

# Lab 10.3: Enterprise RAG Service System Design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.3_enterprise_rag/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.3_enterprise_rag/lab.ipynb)

This lab models the key design decisions for an enterprise RAG service:
model cascade cost/quality tradeoffs, memory budgeting across a heterogeneous fleet,
semantic cache hit rate impact, and latency budget allocation.

In [ ]:
# Install dependencies for plotting and computation
import subprocess
import sys
# Ensure matplotlib and numpy are available
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'numpy'])

In [ ]:
import numpy as np  # numerical computation
import matplotlib.pyplot as plt  # plotting
import matplotlib.patches as mpatches  # custom legend elements

# Configure clean plot defaults
plt.rcParams['figure.figsize'] = (10, 6)  # default figure size
plt.rcParams['font.size'] = 11  # readable text
plt.rcParams['axes.grid'] = True  # gridlines on
plt.rcParams['grid.alpha'] = 0.3  # subtle grid

## Experiment 1: Model Cascade Cost Analysis

The cascade routes simple queries to 8B (cheap) and complex queries to 70B (expensive).
Let's model the blended cost as we vary the routing split.

In [ ]:
# === Parameters: Model costs and routing ===
cost_8b_per_query = 0.0014  # dollars per query on 8B INT8
cost_70b_per_query = 0.022  # dollars per query on 70B INT4
cost_cached_per_query = 0.0001  # dollars per cached query (lookup only)
cache_hit_rate = 0.40  # semantic cache hit rate (40% of queries)

# Sweep the fraction of non-cached queries sent to 8B vs 70B
pct_to_8b = np.linspace(0, 1.0, 50)  # fraction of uncached queries going to 8B
pct_to_70b = 1.0 - pct_to_8b  # remainder goes to 70B

# Calculate blended cost per query
# Total cost = cache_hit_rate * cached_cost + (1-cache_hit_rate) * (pct_8b * cost_8b + pct_70b * cost_70b)
blended_cost = (cache_hit_rate * cost_cached_per_query +  # cached portion
                (1 - cache_hit_rate) * (pct_to_8b * cost_8b_per_query +  # 8B portion
                                        pct_to_70b * cost_70b_per_query))  # 70B portion

# Target cost threshold
target_cost = 0.01  # $0.01 per query target

# Find the minimum 8B routing needed to meet target
meets_target = pct_to_8b[blended_cost <= target_cost]  # find where cost is below target
min_8b_pct = meets_target[0] if len(meets_target) > 0 else 1.0  # minimum 8B fraction needed

# Plot cost curve
fig_c2, (ax1_c2, ax2_c2) = plt.subplots(1, 2, figsize=(14, 5))  # two subplots

# Left: blended cost vs routing split
ax1_c2.plot(pct_to_8b * 100, blended_cost * 1000, color='#2563eb', linewidth=2.5,
         label='Blended cost (with 40% cache)')  # cost in millicents
ax1_c2.axhline(y=target_cost * 1000, color='#991b1b', linestyle='--', linewidth=2,
            label=f'Target: ${target_cost}/query')  # target line
# Shade feasible region
ax1_c2.fill_between(pct_to_8b * 100, 0, blended_cost * 1000,
                 where=blended_cost <= target_cost, alpha=0.2, color='#dcfce7')  # green = under budget
ax1_c2.set_xlabel('% of Non-Cached Queries Routed to 8B')  # x-axis
ax1_c2.set_ylabel('Blended Cost per Query (millicents)')  # y-axis
ax1_c2.set_title('Model Cascade: Cost vs Routing Split')  # title
ax1_c2.legend()  # legend
ax1_c2.set_ylim(0, 15)  # y-range in millicents

# Right: monthly cost at 130M queries/month
monthly_queries = 130_000_000  # ~50 QPS sustained over a month
monthly_costs = blended_cost * monthly_queries  # total monthly spend
budget_limit = 50_000  # $50K monthly budget

ax2_c2.plot(pct_to_8b * 100, monthly_costs / 1000, color='#2563eb', linewidth=2.5)  # cost in $K
ax2_c2.axhline(y=budget_limit / 1000, color='#991b1b', linestyle='--', linewidth=2,
            label=f'Budget: ${budget_limit/1000:.0f}K/month')  # budget line
ax2_c2.fill_between(pct_to_8b * 100, 0, monthly_costs / 1000,
                 where=monthly_costs <= budget_limit, alpha=0.2, color='#dcfce7')  # under budget
ax2_c2.set_xlabel('% of Non-Cached Queries Routed to 8B')  # x-axis
ax2_c2.set_ylabel('Monthly Cost ($K)')  # y-axis
ax2_c2.set_title(f'Monthly Cost at {monthly_queries/1e6:.0f}M Queries/Month')  # title
ax2_c2.legend()  # legend

plt.tight_layout()  # prevent overlap
plt.show()  # render

# Print findings
print(f'\nTo meet ${target_cost}/query target with {cache_hit_rate*100:.0f}% cache hit rate:')
print(f'  Minimum {min_8b_pct*100:.0f}% of non-cached queries must go to 8B')
print(f'  At 50/50 split: ${blended_cost[25]*1000:.1f} millicents/query')
print(f'  At 70/30 (8B/70B): ${blended_cost[35]*1000:.1f} millicents/query')

## Experiment 2: Memory Budget Calculator

Memory planning determines concurrency. Let's calculate GPU memory
allocation for both the 70B and 8B models with different configurations.

In [ ]:
# === Parameters: GPU and model specs ===
gpu_vram_gb = 24  # A10G VRAM in GB

def calculate_memory_budget(model_params_b, quant_bits, num_gpus, num_layers,
                            num_kv_heads, head_dim, max_seq_len, max_concurrent):
    """Calculate memory budget for a model deployment.
    
    Returns dict with component memory usage and feasibility check.
    """
    # Model weights: params * bits / 8 bytes, distributed across GPUs
    weight_memory_gb = (model_params_b * 1e9 * quant_bits / 8) / 1e9 / num_gpus  # GB per GPU
    
    # KV cache: 2 (K and V) * layers * heads * head_dim * 2 bytes (FP16) * seq_len * concurrent / GPUs
    kv_per_token_bytes = 2 * num_layers * num_kv_heads * head_dim * 2  # bytes per token per request
    total_kv_gb = (kv_per_token_bytes * max_seq_len * max_concurrent) / 1e9 / num_gpus  # GB per GPU
    
    # Activations and workspace (estimated as 5-10% of VRAM)
    activations_gb = 2.0  # GB per GPU (fixed estimate)
    
    # CUDA overhead and fragmentation
    cuda_overhead_gb = gpu_vram_gb * 0.10  # 10% of VRAM for CUDA runtime
    
    # Total and feasibility
    total_gb = weight_memory_gb + total_kv_gb + activations_gb + cuda_overhead_gb  # total per GPU
    feasible = total_gb <= gpu_vram_gb  # does it fit?
    
    return {
        'weights_gb': weight_memory_gb,
        'kv_cache_gb': total_kv_gb,
        'activations_gb': activations_gb,
        'cuda_gb': cuda_overhead_gb,
        'total_gb': total_gb,
        'feasible': feasible,
        'headroom_gb': gpu_vram_gb - total_gb  # positive = fits, negative = OOM
    }

# 70B INT4 on 4xA10G: 80 layers, 8 KV heads (GQA), 128 head_dim
budget_70b = calculate_memory_budget(
    model_params_b=70, quant_bits=4, num_gpus=4,
    num_layers=80, num_kv_heads=8, head_dim=128,
    max_seq_len=4096, max_concurrent=4  # 4 concurrent at 4K context
)

# 8B INT8 on 1xA10G: 32 layers, 8 KV heads (GQA), 128 head_dim
budget_8b = calculate_memory_budget(
    model_params_b=8, quant_bits=8, num_gpus=1,
    num_layers=32, num_kv_heads=8, head_dim=128,
    max_seq_len=4096, max_concurrent=16  # 16 concurrent at 4K context
)

# Visualize memory budgets
fig_c3, (ax1_c3, ax2_c3) = plt.subplots(1, 2, figsize=(14, 5))  # side by side

# Helper to plot stacked bar
def plot_memory_budget(ax, budget, title, gpu_vram):
    """Plot a stacked bar chart of memory allocation."""
    components = ['Weights', 'KV Cache', 'Activations', 'CUDA']  # memory components
    values = [budget['weights_gb'], budget['kv_cache_gb'],
              budget['activations_gb'], budget['cuda_gb']]  # sizes
    colors = ['#dbeafe', '#dcfce7', '#f3e8ff', '#f3f4f6']  # pastel colors
    
    # Stacked horizontal bar
    left = 0  # starting position for each segment
    for comp, val, col in zip(components, values, colors):
        ax.barh(0, val, left=left, color=col, edgecolor='#000',
                linewidth=1, label=f'{comp}: {val:.1f} GB')  # draw segment
        # Label inside segment if big enough
        if val > 1.5:  # only label segments > 1.5 GB
            ax.text(left + val/2, 0, f'{val:.1f}', ha='center', va='center', fontsize=10)
        left += val  # advance position
    
    # Draw VRAM limit
    ax.axvline(x=gpu_vram, color='#991b1b', linestyle='--', linewidth=2,
               label=f'VRAM Limit: {gpu_vram} GB')  # capacity line
    # Color the status
    status = 'FITS' if budget['feasible'] else 'OOM'
    status_color = '#166534' if budget['feasible'] else '#991b1b'
    ax.text(gpu_vram + 0.3, 0, f'{status}\n({budget["headroom_gb"]:.1f} GB)',
            va='center', fontweight='bold', color=status_color)  # status label
    
    ax.set_xlim(0, gpu_vram + 5)  # x range
    ax.set_yticks([])  # hide y-axis (single bar)
    ax.set_xlabel('GPU Memory (GB)')  # x label
    ax.set_title(title)  # chart title
    ax.legend(loc='upper right', fontsize=9)  # legend

plot_memory_budget(ax1_c3, budget_70b, '70B INT4 per GPU (4xA10G, 4 concurrent, 4K ctx)', gpu_vram_gb)
plot_memory_budget(ax2_c3, budget_8b, '8B INT8 per GPU (1xA10G, 16 concurrent, 4K ctx)', gpu_vram_gb)

plt.tight_layout()  # prevent overlap
plt.show()  # render

# Print summaries
print('\n=== 70B INT4 (per GPU, 4xA10G) ===')
print(f'  Weights: {budget_70b["weights_gb"]:.1f} GB | KV: {budget_70b["kv_cache_gb"]:.1f} GB | Total: {budget_70b["total_gb"]:.1f}/{gpu_vram_gb} GB')
print(f'  Max concurrent: 4 requests at 4K context')
print(f'\n=== 8B INT8 (1xA10G) ===')
print(f'  Weights: {budget_8b["weights_gb"]:.1f} GB | KV: {budget_8b["kv_cache_gb"]:.1f} GB | Total: {budget_8b["total_gb"]:.1f}/{gpu_vram_gb} GB')
print(f'  Max concurrent: 16 requests at 4K context')

## Experiment 3: Cache Hit Rate Impact on Latency

Semantic caching eliminates the generation step entirely for repeated queries.
Let's model how cache hit rate affects the P50 and P99 latency.

In [ ]:
# === Parameters: Pipeline stage latencies (ms) ===
latency_embed_ms = 50  # embedding generation time
latency_search_ms = 100  # vector + BM25 hybrid search
latency_rerank_ms = 150  # cross-encoder reranking
latency_assembly_ms = 20  # context assembly
latency_gen_8b_ms = 500  # 8B model generation
latency_gen_70b_ms = 1500  # 70B model generation
latency_postprocess_ms = 80  # post-processing
latency_cache_lookup_ms = 5  # semantic cache lookup

# Full pipeline latency (no cache hit)
full_pipeline_8b = (latency_embed_ms + latency_search_ms + latency_rerank_ms +
                    latency_assembly_ms + latency_gen_8b_ms + latency_postprocess_ms)  # total for 8B
full_pipeline_70b = (latency_embed_ms + latency_search_ms + latency_rerank_ms +
                     latency_assembly_ms + latency_gen_70b_ms + latency_postprocess_ms)  # total for 70B

# Cache hit latency (skip retrieval + generation entirely)
cache_hit_latency = latency_cache_lookup_ms  # just the lookup

# Sweep cache hit rates
cache_rates = np.linspace(0, 0.8, 40)  # 0% to 80% cache hit rate

# Model: P50 latency = weighted average (cache hits pull average down)
# With 70/30 routing split among non-cached queries
pct_8b_of_uncached = 0.70  # 70% of non-cached go to 8B
pct_70b_of_uncached = 0.30  # 30% of non-cached go to 70B

# Weighted uncached latency
uncached_avg_latency = (pct_8b_of_uncached * full_pipeline_8b +
                        pct_70b_of_uncached * full_pipeline_70b)  # blended uncached latency

# P50 approximation: blend of cache hit and uncached latencies
p50_latency = cache_rates * cache_hit_latency + (1 - cache_rates) * uncached_avg_latency  # weighted average

# P99: even with high cache rate, some queries will be complex + cache miss
# P99 is dominated by 70B generation on complex queries
p99_latency = full_pipeline_70b * np.ones_like(cache_rates)  # P99 always hits worst case
# But at very high cache rates, P99 shifts toward 8B pipeline
p99_latency = np.where(cache_rates > 0.7, full_pipeline_8b + 200, p99_latency)  # at >70% cache, P99 improves

# Plot
fig_c4, ax_c4 = plt.subplots(figsize=(10, 6))  # create figure

ax_c4.plot(cache_rates * 100, p50_latency, color='#2563eb', linewidth=2.5, label='P50 Latency')  # P50 curve
ax_c4.plot(cache_rates * 100, p99_latency, color='#991b1b', linewidth=2, linestyle='--', label='P99 Latency')  # P99 curve

# Draw SLO targets
ax_c4.axhline(y=1500, color='#166534', linestyle=':', linewidth=1.5, label='P50 SLO (1.5s)')  # P50 target
ax_c4.axhline(y=3000, color='#64748b', linestyle=':', linewidth=1.5, label='P99 SLO (3.0s)')  # P99 target

# Shade the feasible region (under SLO)
ax_c4.fill_between(cache_rates * 100, 0, p50_latency,
                where=p50_latency <= 1500, alpha=0.15, color='#dcfce7')  # green = meets P50 SLO

# Annotations
ax_c4.annotate(f'40% cache: {p50_latency[20]:.0f}ms P50',
            xy=(40, p50_latency[20]), xytext=(50, p50_latency[20] + 200),
            arrowprops=dict(arrowstyle='->', color='#2563eb'),
            fontsize=10, color='#2563eb')  # annotate the 40% point

ax_c4.set_xlabel('Semantic Cache Hit Rate (%)')  # x-axis
ax_c4.set_ylabel('End-to-End Latency (ms)')  # y-axis
ax_c4.set_title('Enterprise RAG: Latency vs Cache Hit Rate')  # title
ax_c4.legend(loc='upper right')  # legend
ax_c4.set_ylim(0, 3500)  # y range

plt.tight_layout()  # prevent clipping
plt.show()  # render

# Print key findings
print(f'\n=== Latency Analysis ===')
print(f'Full pipeline (8B): {full_pipeline_8b}ms')
print(f'Full pipeline (70B): {full_pipeline_70b}ms')
print(f'Cache hit: {cache_hit_latency}ms')
print(f'\nAt 40% cache hit rate: P50 = {p50_latency[20]:.0f}ms (target: <1500ms)')
print(f'Generation accounts for {latency_gen_70b_ms/full_pipeline_70b*100:.0f}% of 70B pipeline latency')

## Experiment 4: Monthly Cost Breakdown

Visualize where the $14.4K/month budget goes across
the heterogeneous fleet (generation, retrieval, storage, batch).

In [ ]:
# === Parameters: Monthly cost components ===
cost_components = {
    '70B Generation\n(g5.12xlarge)': 6208,  # 1.5 avg instances
    '8B Generation\n(2x g5.xlarge)': 1475,  # 2 replicas
    'Embedding +\nReranker': 738,  # shared GPU
    'Vector DB\n(OpenSearch)': 2847,  # 3-node cluster
    'Batch Indexing\n(spot)': 182,  # nightly spot instances
    'API + Network\n+ Storage': 1095,  # infrastructure overhead
    'Contingency\n(15%)': 1877,  # buffer for spikes
}

# Total monthly cost
total_monthly = sum(cost_components.values())  # sum all components

# Create visualization
fig_c5, (ax1_c5, ax2_c5) = plt.subplots(1, 2, figsize=(14, 6))  # two charts

# Left: horizontal bar chart of costs
labels = list(cost_components.keys())  # component names
values = list(cost_components.values())  # dollar amounts
bar_colors = ['#fef3c7', '#dcfce7', '#dbeafe', '#f3e8ff', '#ffedd5', '#f3f4f6', '#ffe4e6']  # pastel colors

bars = ax1_c5.barh(labels, values, color=bar_colors, edgecolor='#000', linewidth=1)  # horizontal bars
# Label each bar with dollar amount
for bar, val in zip(bars, values):
    ax1_c5.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=10)  # dollar label

ax1_c5.set_xlabel('Monthly Cost ($)')  # x-axis
ax1_c5.set_title(f'Enterprise RAG: Monthly Cost Breakdown (${total_monthly:,.0f} total)')  # title
ax1_c5.set_xlim(0, max(values) * 1.3)  # room for labels

# Right: cost per query breakdown by scenario
scenarios = ['All 70B\n(no optimization)', 'Cascade\n(70/30 8B/70B)', 'Cascade +\n40% Cache', 'Cascade +\n55% Cache']  # scenarios
# Cost per query for each scenario
cpq_all_70b = cost_70b_per_query  # everything goes to 70B
cpq_cascade = 0.70 * cost_8b_per_query + 0.30 * cost_70b_per_query  # cascade without cache
cpq_cascade_40 = 0.40 * cost_cached_per_query + 0.60 * cpq_cascade  # 40% cache
cpq_cascade_55 = 0.55 * cost_cached_per_query + 0.45 * cpq_cascade  # 55% cache
cpq_values = [cpq_all_70b * 1000, cpq_cascade * 1000, cpq_cascade_40 * 1000, cpq_cascade_55 * 1000]  # millicents

scenario_colors = ['#ffe4e6', '#fef3c7', '#dbeafe', '#dcfce7']  # red to green progression
bars2 = ax2_c5.bar(scenarios, cpq_values, color=scenario_colors, edgecolor='#000', linewidth=1.2)  # bar chart

# Target line
ax2_c5.axhline(y=10, color='#991b1b', linestyle='--', linewidth=2,
            label='Target: $0.01/query')  # $0.01 = 10 millicents

# Label bars
for bar, val in zip(bars2, cpq_values):  # add value labels
    ax2_c5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'${val/1000:.4f}', ha='center', fontsize=9, fontweight='bold')  # dollar amount

ax2_c5.set_ylabel('Cost per Query (millicents)')  # y-axis
ax2_c5.set_title('Cost Optimization: Impact of Cascade + Caching')  # title
ax2_c5.legend()  # legend

plt.tight_layout()  # prevent overlap
plt.show()  # render

# Print summary
print(f'\n=== Cost Optimization Summary ===')
print(f'All 70B (no optimization): ${cpq_all_70b*1000:.1f} millicents/query')
print(f'Cascade only: ${cpq_cascade*1000:.1f} millicents/query ({(1-cpq_cascade/cpq_all_70b)*100:.0f}% savings)')
print(f'Cascade + 40% cache: ${cpq_cascade_40*1000:.1f} millicents/query ({(1-cpq_cascade_40/cpq_all_70b)*100:.0f}% savings)')
print(f'Cascade + 55% cache: ${cpq_cascade_55*1000:.1f} millicents/query ({(1-cpq_cascade_55/cpq_all_70b)*100:.0f}% savings)')
print(f'\nTotal monthly cost: ${total_monthly:,.0f} (well under $50K budget)')